# Which change should the peanut-sheller designer make first?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmudrc/design-research-workshop/blob/workshop-three-agent-demos/notebooks/peanut_sheller_propose_critic.ipynb)

A small grower wants a hand-operated tabletop sheller. A proposer specifies the mechanisms. A critic checks the design and assigns changes. Give the proposer one change per round and see which features make it into the next draft.

Run the numbered sections in order. A CPU runtime is enough. The workshop PyBot uses local Python rules and runs without an API key.

## 1. Set up
Run the install cell once in Colab. If you installed the workshop package locally, start in the next section.

In [ ]:
# @title Install the workshop package { display-mode: "form" }
%pip -q install "design-research-workshop @ git+https://github.com/cmudrc/design-research-workshop.git@workshop-three-agent-demos"

## 2. Choose a starting design

Pick a way to feed pods, crack shells, and separate the kernels. The initial design has a fixed cracking clearance and fixed cleaning access. Select **start with adjustable clearance** to give the critic a more developed concept.

In [1]:
# @title Starting concept { display-mode: "form" }
import json
from dataclasses import replace

from design_research.problems import get_problem
from design_research_workshop import workshop_client
from design_research_workshop.peanut import (
    BRIEF, PROBLEM_ID, PROTECTIONS, REVIEW_CRITERIA,
    ShellerConcept, sheller_pattern, sheller_task,
)

feed = "Gravity hopper"  # @param ["Hand-fed tray", "Gravity hopper", "Indexed feed wheel"]
crack = "Hand-cranked rollers"  # @param ["Hinged lever press", "Hand-cranked rollers", "Sliding rub plates"]
separate = "Hand-shaken sieve"  # @param ["Tilted screen", "Hand-shaken sieve", "Hand-cranked air separator"]
start_with_adjustable_clearance = False  # @param {type:"boolean"}

concept = ShellerConcept(feed, crack, separate)
if start_with_adjustable_clearance:
    concept = replace(concept, kernel_protection=PROTECTIONS[crack])

print(BRIEF)
print(f"Adapted from the problem library's “{get_problem(PROBLEM_ID).metadata.title}” brief.")
print(f"\nStarting concept: {feed} → {crack} → {separate}")
print("Cracking clearance:", concept.kernel_protection)
print("Cleaning access:", concept.cleaning_access)

Design a non-electric tabletop peanut sheller for a small grower. Aim to keep kernels intact, use readily available materials, and make food-contact parts easy to clean.
Adapted from the problem library's “Device to shell peanuts” brief.

Starting concept: Gravity hopper → Hand-cranked rollers → Hand-shaken sieve
Cracking clearance: Fixed clearance with no size adjustment during use.
Cleaning access: Fixed separation assembly with access from the outlet.


## 3. Set the critic's priorities and run the loop

The critic looks for specified design features. It can recognize an adjustable cracking limit, access to the separator and tray for cleaning, and a way to meter incoming pods. Choose the criteria and their order. Predict which provision you will see in round two.

In [2]:
# @title Review assignment { display-mode: "form" }
review_order = "kernel protection → cleaning access"  # @param ["kernel protection → cleaning access", "cleaning access → kernel protection", "feed control → cleaning access → kernel protection", "cleaning access"]
changes_per_revision = 1  # @param {type:"slider", min:1, max:3, step:1}
max_iterations = 3  # @param {type:"slider", min:1, max:4, step:1}
criteria = review_order.split(" → ")
task = sheller_task(concept, criteria=criteria, changes_per_revision=changes_per_revision)

with workshop_client() as client:
    result = sheller_pattern(client, max_iterations=max_iterations).run(task)

print(f"Review order: {review_order}")
print(f"Rounds used: {result.iterations}/{max_iterations}")
print(f"Stopping reason: {result.terminated_reason}")

Review order: kernel protection → cleaning access
Rounds used: 3/3
Stopping reason: approved


## 4. Inspect the revision trace

Read the proposal and assigned goals for each round. Which change follows directly from the preceding critique? Where did the proposer keep an earlier decision?

In [3]:
for event in result.critique_iterations:
    draft = json.loads(event["proposal"])
    print(f"Round {event['iteration']}")
    for name in criteria:
        rule = REVIEW_CRITERIA[name]
        print(f"  {rule.label}: {draft[rule.field]}")
    print(f"  Critic: {event['feedback'] or 'Approved'}")
    for goal in event["revision_goals"]:
        print(f"    {goal}")
    print()

Round 1
  Kernel protection: Fixed clearance with no size adjustment during use.
  Cleaning access: Fixed separation assembly with access from the outlet.
  Critic: Kernel protection needs attention. Specify how the operator can limit compression for different pod sizes. Cleaning access needs attention. Specify access to both the separator and its collection tray.
    Change kernel protection: Adjustable roller gap and a compliant roller surface.

Round 2
  Kernel protection: Adjustable roller gap and a compliant roller surface.
  Cleaning access: Fixed separation assembly with access from the outlet.
  Critic: Cleaning access needs attention. Specify access to both the separator and its collection tray.
    Keep kernel protection: Adjustable roller gap and a compliant roller surface.
    Change cleaning access: Unclip the sieve and empty the removable collection tray.

Round 3
  Kernel protection: Adjustable roller gap and a compliant roller surface.
  Cleaning access: Unclip the siev

## 5. Compare priorities with the same design budget

Give each team two rounds and one change between rounds. Both teams start with the three mechanisms you selected and the original fixed provisions. Team A reviews kernel protection first. Team B reviews cleaning access first. Which features does each team carry into round two?

In [4]:
comparison_seed = ShellerConcept(feed, crack, separate)
with workshop_client() as client:
    for team, priority in [("A", "kernel protection"), ("B", "cleaning access")]:
        task = sheller_task(comparison_seed, priority, changes_per_revision=1)
        trial = sheller_pattern(client, max_iterations=2).run(task)
        last = trial.critique_iterations[-1]
        final_design = json.loads(last["proposal"])
        print(f"Team {team} prioritized {priority}.")
        print("  Kernel protection:", final_design["kernel_protection"])
        print("  Cleaning access:", final_design["cleaning_access"])
        print("  Remaining feedback:", last["feedback"] or "Approved")
        print()

Team A prioritized kernel protection.
  Kernel protection: Adjustable roller gap and a compliant roller surface.
  Cleaning access: Fixed separation assembly with access from the outlet.
  Remaining feedback: Cleaning access needs attention. Specify access to both the separator and its collection tray.

Team B prioritized cleaning access.
  Kernel protection: Fixed clearance with no size adjustment during use.
  Cleaning access: Unclip the sieve and empty the removable collection tray.
  Remaining feedback: Kernel protection needs attention. Specify how the operator can limit compression for different pod sizes.



## Discuss

- Which first change would you choose for a grower who processes a mix of pod sizes? What information would change your choice?
- The critic approves a written provision. What would you measure on a prototype to judge kernel damage or ease of cleaning? Specify a comparison and an observation that would make you revise the design again.